In [ ]:
# -*- coding: utf-8 -*-

Untitled6.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/18JE36eNDrgSwWXsmvUQbAWjw-hsD1n6z

prime due celle servono a scaricare set totali di dati si runnano solo prima volta poi vengono creati due subset che rimangono fissi per utta la durata delle prove per riproducbilita dei risultatai.

In [2]:
#from google.colab import drive
#import os

# 1. Monta Google Drive
#drive.mount('/content/drive')

# 2. Definisci le directory base sul tuo Drive
#base_dir = '/content/drive/MyDrive/set dati machine learning'
#save_dir = os.path.join(base_dir, 'processed_data')

import os

base_dir = r'C:\Percorso\Della\Tua\Cartella\set dati machine learning'

save_dir = os.path.join(base_dir, 'processed_data')
os.makedirs(save_dir, exist_ok=True)

# Crea la cartella per i dati processati se non esiste già
#os.makedirs(save_dir, exist_ok=True)
print(f"Cartella di lavoro impostata su: {base_dir}")

# Installiamo il plugin per leggere i file compressi da Pandas
!pip install hdf5plugin

import numpy as np
import h5py
import hdf5plugin # Importante: abilita la decompressione!
import os

# Percorsi dei file originali sul tuo Drive
bg_file_path = '/content/drive/MyDrive/set dati machine learning/events_LHCO2020_backgroundMC_Pythia.h5'
blackbox_file_path = '/content/drive/MyDrive/set dati machine learning/events_LHCO2020_BlackBox1.h5'

# Percorsi di salvataggio
save_dir = '/content/drive/MyDrive/set dati machine learning/processed_data'
bg_save_path = os.path.join(save_dir, 'background_subset.npy')
test_save_path = os.path.join(save_dir, 'blackbox_subset.npy')

os.makedirs(save_dir, exist_ok=True)

def fast_smart_sample_h5(file_path, n_samples):
    print(f"\nApertura intelligente di {file_path.split('/')[-1]}...")

    with h5py.File(file_path, 'r') as f:
        key = list(f.keys())[0]
        node = f[key]

        if isinstance(node, h5py.Group):
            print(f"Trovato il gruppo Pandas '{key}'. Cerco l'array dei dati interni...")
            if 'block0_values' in node.keys():
                dataset = node['block0_values']
            elif 'table' in node.keys():
                dataset = node['table']
            else:
                raise KeyError("Struttura non riconosciuta.")
        else:
            dataset = node

        shape = dataset.shape
        print(f"Dimensione dell'array trovato su disco: {shape}")

        if shape[0] > shape[1]:
            total_events = shape[0]
            is_transposed = False
        else:
            total_events = shape[1]
            is_transposed = True

        print(f"Totale eventi nel file: {total_events}")

        if n_samples >= total_events:
            print("Caricamento totale...")
            data = dataset[:]
            if is_transposed:
                data = data.T
            return data.astype(np.float32)

        print(f"Estrazione del blocco di {n_samples} eventi dal disco (istantaneo e salva RAM)...")

        # ESTRAZIONE IN BLOCCO: Veloce e amica della RAM su file compressi!
        if is_transposed:
            subset_data = dataset[:, :n_samples].T
        else:
            subset_data = dataset[:n_samples, :]

        return subset_data.astype(np.float32)

# ==========================================
# ESECUZIONE DELL'ESTRAZIONE
# ==========================================

if not os.path.exists(bg_save_path):
    bg_data = fast_smart_sample_h5(bg_file_path, n_samples=100000)
    np.save(bg_save_path, bg_data)
    print(f"Sottoinsieme di background salvato con successo! (Dimensioni in RAM: {bg_data.nbytes / 1e6:.2f} MB)")
    del bg_data
else:
    print(f"File {bg_save_path.split('/')[-1]} già esistente. Salto l'estrazione.")

if not os.path.exists(test_save_path):
    test_data = fast_smart_sample_h5(blackbox_file_path, n_samples=20000)
    np.save(test_save_path, test_data)
    print(f"Sottoinsieme BlackBox salvato con successo! (Dimensioni in RAM: {test_data.nbytes / 1e6:.2f} MB)")
    del test_data
else:
    print(f"File {test_save_path.split('/')[-1]} già esistente. Salto l'estrazione.")

print("\nOperazione completata! RAM al sicuro.")

Cartella di lavoro impostata su: C:\Percorso\Della\Tua\Cartella\set dati machine learning
   ---------------------------------------- 0.0/3.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.4 MB 960.0 kB/s eta 0:00:04
   - -------------------------------------- 0.1/3.4 MB 1.2 MB/s eta 0:00:03
   -- ------------------------------------- 0.2/3.4 MB 2.0 MB/s eta 0:00:02
   --- ------------------------------------ 0.3/3.4 MB 1.9 MB/s eta 0:00:02
   ------ --------------------------------- 0.6/3.4 MB 2.4 MB/s eta 0:00:02
   ------- -------------------------------- 0.7/3.4 MB 2.5 MB/s eta 0:00:02
   ---------- ----------------------------- 0.9/3.4 MB 2.7 MB/s eta 0:00:01
   ----------- ---------------------------- 1.0/3.4 MB 2.7 MB/s eta 0:00:01
   ------------- -------------------------- 1.1/3.4 MB 2.8 MB/s eta 0:00:01
   ---------------- ----------------------- 1.4/3.4 MB 3.1 MB/s eta 0:00:01
   ------------------ --------------------- 1.5/3.4 MB 3.1 MB/s eta 0:00:01
   

FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = '/content/drive/MyDrive/set dati machine learning/events_LHCO2020_backgroundMC_Pythia.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

successiva cella metto le cose in numpy e vedo grandezza dei subset e normalizzo dati poi li ordino in base al valore cosi che l rete neurale si possa allenare in maniera significativa. senza non si va avnati e le anomalie vengono perche messe intorno alle 0 perche non ricostruite dle tutto. vengono tratatte solo prime 100 PARTICELLE PER TOGLIERE GLI ZERI E ALLEGGERIRE IL TUTTO IN MANIERA ANALOGA AL PAPER, inltre si settano a 0 gli angoli della prima particella e esegue un arotazione per tutto la riga cosi che gli angoli abbiano senso senno sono totalmente random

In [ ]:
import numpy as np
from sklearn.preprocessing import MaxAbsScaler

print("1. Caricamento dei dati grezzi in memoria...")
X_bg = np.load('/content/drive/MyDrive/set dati machine learning/processed_data/background_subset.npy')
X_test = np.load('/content/drive/MyDrive/set dati machine learning/processed_data/blackbox_subset.npy')

def preprocess_fisico_completo(dati, max_particelle=100):
    

    1. Ordina per pT decrescente.
    2. CENTRA gli angoli (eta, phi) sulla particella più energetica.
    3. Taglia alle top 'max_particelle'.
    

In [ ]:
    N_eventi, N_feature = dati.shape
    N_particelle_totali = N_feature // 3
    dati_3d = dati.reshape(N_eventi, N_particelle_totali, 3)

    # 1. ORDINAMENTO PER pT
    indici_ordine = np.argsort(dati_3d[:, :, 0], axis=1)[:, ::-1]
    dati_3d = np.take_along_axis(dati_3d, indici_ordine[:, :, np.newaxis], axis=1)

    # 2. CENTRAMENTO GEOMETRICO
    # Prendiamo le coordinate della prima particella (la più energetica) di ogni evento
    eta_centro = dati_3d[:, 0, 1][:, np.newaxis]
    phi_centro = dati_3d[:, 0, 2][:, np.newaxis]

    # Creiamo una maschera per non toccare i "buchi" vuoti (lo zero padding)
    # dove il pT è esattamente 0
    maschera_vere = (dati_3d[:, :, 0] > 0).astype(float)

    # Sottraiamo il centro per ottenere DeltaEta
    dati_3d[:, :, 1] = (dati_3d[:, :, 1] - eta_centro) * maschera_vere

    # Sottraiamo il centro per ottenere DeltaPhi, e forziamo il risultato tra -pi e pi
    d_phi = dati_3d[:, :, 2] - phi_centro
    d_phi = (d_phi + np.pi) % (2 * np.pi) - np.pi
    dati_3d[:, :, 2] = d_phi * maschera_vere

    # 3. TAGLIO E APPIATTIMENTO
    taglio = min(max_particelle, N_particelle_totali)
    dati_finali = dati_3d[:, :taglio, :].reshape(N_eventi, taglio * 3)

    return dati_finali

print("2. Elaborazione (Ordine + Centramento + Taglio a 100) sul Background...")
X_bg_processed = preprocess_fisico_completo(X_bg, max_particelle=100)

print("3. Elaborazione (Ordine + Centramento + Taglio a 100) sulla BlackBox...")
X_test_processed = preprocess_fisico_completo(X_test, max_particelle=100)

print("\n4. Normalizzazione (MaxAbsScaler)...")
scaler = MaxAbsScaler()
X_bg_scaled = scaler.fit_transform(X_bg_processed)
X_test_scaled = scaler.transform(X_test_processed)

print(f"\n---> Dimensioni finali background: {X_bg_scaled.shape}")
print("---> Pronto per addestrare l'Autoencoder!")

passiamo a pytorch e split in train e val per entrambi i due dataset  nomalizzazione del set con anche il segnale in cella 5. valori su cui si puo lavoarre è batch size e quanto grande vali e test

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# 1. Split 70% Train - 30% Validation
X_train, X_val = train_test_split(X_bg_scaled, test_size=0.30, random_state=42)

# 2. Conversione in Tensori PyTorch (Float32)
tensor_X_train = torch.tensor(X_train, dtype=torch.float32)
tensor_X_val = torch.tensor(X_val, dtype=torch.float32)

# 3. TensorDataset (Input, Target) -> per l'Autoencoder l'input è il target (X, X)
train_dataset = TensorDataset(tensor_X_train, tensor_X_train)
val_dataset = TensorDataset(tensor_X_val, tensor_X_val)

# 4. Creazione DataLoader
batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Batch nel Train Loader: {len(train_loader)}")
print(f"Batch nel Val Loader: {len(val_loader)}")

# 1. Conversione in Tensori PyTorch (Float32)
tensor_X_test = torch.tensor(X_test_scaled, dtype=torch.float32)

# 2. TensorDataset per il test
test_dataset = TensorDataset(tensor_X_test, tensor_X_test)

# 3. Creazione DataLoader
# Niente shuffle qui: vogliamo mantenere l'ordine originale degli eventi
# per mappare correttamente gli anomaly score e calcolare le masse in seguito!
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Batch nel Test Loader (BlackBox): {len(test_loader)}")
print("\nTutti i DataLoader sono pronti! Il prossimo passo è definire l'architettura dell'Autoencoder.")

prima bozza dell'autoencoder poi si lavorera sulle dimensioni dei layer in mezzo, per layer finale con cui costriamo spazio latente il range di modifica è circa 16-4

In [ ]:
import torch
import torch.nn as nn

class DeepConvAutoencoder(nn.Module):
    def __init__(self, num_particelle=100, num_features=3, latent_dim=8):
        super(DeepConvAutoencoder, self).__init__()

        self.num_particelle = num_particelle
        self.num_features = num_features

        # --- ENCODER: CNN Multistrato ---
        self.encoder_cnn = nn.Sequential(
            # 1° Strato: Da 3 feature a 16 filtri
            nn.Conv1d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2),
            nn.MaxPool1d(kernel_size=2), # Dimezza: 100 -> 50

            # 2° Strato: Da 16 a 32 filtri
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            nn.MaxPool1d(kernel_size=2), # Dimezza: 50 -> 25

            # 3° Strato: Da 32 a 64 filtri (Maggiore profondità)
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2)
            # Lunghezza rimane 25. Spazio totale: 64 * 25 = 1600
        )

        self.encoder_linear = nn.Sequential(
            nn.Linear(64 * 25, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, latent_dim) # Compressione finale a 8D
        )

        # --- DECODER: Ricostruzione speculare ---
        self.decoder_linear = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64 * 25),
            nn.BatchNorm1d(64 * 25),
            nn.LeakyReLU(0.2)
        )

        self.decoder_cnn = nn.Sequential(
            # 1° Strato Decoder: 64 -> 32
            nn.ConvTranspose1d(in_channels=64, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),

            # 2° Strato Decoder: 32 -> 16 (Allarga: 25 -> 50)
            nn.ConvTranspose1d(in_channels=32, out_channels=16, kernel_size=2, stride=2),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2),

            # 3° Strato Decoder: 16 -> 3 (Allarga: 50 -> 100)
            nn.ConvTranspose1d(in_channels=16, out_channels=3, kernel_size=2, stride=2)
            # Niente attivazione finale per prevedere valori negativi/positivi scalati
        )

    def forward(self, x):
        batch_size = x.size(0)

        # Rimodella per Conv1d: (Batch, Canali=3, Lunghezza=100)
        x = x.view(batch_size, self.num_particelle, self.num_features).permute(0, 2, 1)

        # Encoding
        x = self.encoder_cnn(x)
        x = x.view(batch_size, -1) # Appiattisce
        latent = self.encoder_linear(x)

        # Decoding
        x = self.decoder_linear(latent)
        x = x.view(batch_size, 64, 25) # Ricrea i canali per ConvTranspose
        x = self.decoder_cnn(x)

        # Ritorna piatto: (Batch, 300)
        x = x.permute(0, 2, 1).contiguous().view(batch_size, -1)

        return x

    def get_latent(self, x):
        

Estrae solo lo spazio 8D per il K-Means

In [ ]:
        batch_size = x.size(0)
        x = x.view(batch_size, self.num_particelle, self.num_features).permute(0, 2, 1)
        x = self.encoder_cnn(x)
        x = x.view(batch_size, -1)
        return self.encoder_linear(x)

# Avvio del modello
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepConvAutoencoder(num_particelle=100, num_features=3, latent_dim=8).to(device)
print(f"Modello Multi-layer CNN creato su: {device}")

allemento modello da fare funzione di salvataggio del migliore e aggiungerlo

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR # <-- Importiamo lo Scheduler
import matplotlib.pyplot as plt

# 1. Configurazione del dispositivo (Cerca la GPU su Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Addestramento in esecuzione su: {device}")

# Spostiamo il modello sul dispositivo scelto
model = model.to(device)

# 2. Definizione di Loss, Ottimizzatore e Scheduler
criterion = nn.MSELoss()
# Partiamo con un LR iniziale del 10% (0.1)
optimizer = optim.Adam(model.parameters(), lr=0.1)

# Impostiamo lo Scheduler: ogni step_size (10 epoche), moltiplica il LR per gamma (1/3)
scheduler = StepLR(optimizer, step_size=10, gamma=1.0/3.0)

# Parametri di addestramento
epochs = 50
train_losses = []
val_losses = []

print("\nInizio addestramento dell'Autoencoder...\n")

# 3. Ciclo di addestramento
for epoch in range(epochs):
    # --- FASE DI TRAINING ---
    model.train() # Mettiamo il modello in modalità addestramento
    running_train_loss = 0.0

    for batch_x, _ in train_loader:
        batch_x = batch_x.to(device)

        # Forward pass: l'autoencoder comprime e ricostruisce
        reconstructed = model(batch_x)

        # Calcolo dell'errore (Loss)
        loss = criterion(reconstructed, batch_x)

        # Backward pass e ottimizzazione
        optimizer.zero_grad() # Azzeriamo i gradienti precedenti
        loss.backward()       # Calcoliamo i nuovi gradienti
        optimizer.step()      # Aggiorniamo i pesi della rete

        running_train_loss += loss.item() * batch_x.size(0)

    # Calcolo loss media dell'epoca
    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    # --- FASE DI VALIDATION ---
    model.eval() # Mettiamo il modello in modalità valutazione
    running_val_loss = 0.0

    with torch.no_grad(): # Spegniamo il calcolo dei gradienti per risparmiare memoria
        for batch_x, _ in val_loader:
            batch_x = batch_x.to(device)
            reconstructed = model(batch_x)
            loss = criterion(reconstructed, batch_x)
            running_val_loss += loss.item() * batch_x.size(0)

    # Calcolo loss media di validazione
    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    # --- AGGIORNAMENTO LEARNING RATE ---
    # Lo scheduler fa un "passo" alla fine di ogni epoca per capire se deve tagliare il LR
    scheduler.step()

    # Stampa i progressi ogni 2 epoche (mostrando anche il LR attuale)
    current_lr = scheduler.get_last_lr()[0]
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Epoca [{epoch+1:2d}/{epochs}] | LR: {current_lr:.5f} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

print("\nAddestramento completato!")

# 4. Salvataggio del modello addestrato nel tuo Drive
model_save_path = '/content/drive/MyDrive/set dati machine learning/processed_data/autoencoder_weights.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Pesi del modello salvati in: {model_save_path}")

# ==========================================
# PLOT DELLA LOSS
# ==========================================
plt.figure(figsize=(10, 6))
plt.plot(range(1, epochs+1), train_losses, label='Training Loss', linewidth=2)
plt.plot(range(1, epochs+1), val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoche', fontsize=12)
plt.ylabel('Mean Squared Error (Loss)', fontsize=12)
plt.title('Curva di Apprendimento dell\'Autoencoder', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

prima rappresentazioni componenti spazio latente e ricostruzione, la ricostruzione non è un granche perche non è il nostro obbiettivo vogliamo anomaly detection. piu ricostrisco bene meno sono bravoa fare anomaly detection, per questo prima validazione cosi bassa

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Mettiamo il modello in modalità valutazione
model.eval()

# Prendiamo un intero blocco di validazione per fare i grafici
sample_data, _ = next(iter(val_loader))
sample_data = sample_data.to(device)

with torch.no_grad():
    # 1. Passiamo i dati usando la funzione corretta!
    latent_space = model.get_latent(sample_data)

    # 2. Passiamo i dati in tutto l'Autoencoder per la ricostruzione
    reconstructed_data = model(sample_data)

# Portiamo i dati sulla CPU e li convertiamo in Numpy per i grafici
sample_data_np = sample_data.cpu().numpy()
latent_space_np = latent_space.cpu().numpy()
reconstructed_data_np = reconstructed_data.cpu().numpy()

# ... (Il resto del codice per i PLOT rimane uguale) ...
# ==========================================
# PLOT 1: LO SPAZIO LATENTE (Le 8 Dimensioni)
# ==========================================
plt.figure(figsize=(16, 8))
plt.suptitle("Distribuzione delle 8 variabili nello Spazio Latente (Background)", fontsize=16, fontweight='bold')

for i in range(8):
    plt.subplot(2, 4, i+1)
    # Disegniamo l'istogramma per ogni singola dimensione latente
    plt.hist(latent_space_np[:, i], bins=50, color='teal', alpha=0.7, edgecolor='black')
    plt.title(f"Dimensione Latente {i+1}")
    plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

# ==========================================
# PLOT 2: ORIGINALE vs RICOSTRUITO
# ==========================================
# Prendiamo il primissimo evento del batch e mostriamo solo le prime 60 feature (su 2100)
evento_idx = 0
num_features = 60

plt.figure(figsize=(15, 5))
plt.plot(sample_data_np[evento_idx, :num_features], label='Dato Originale (Input)', color='blue', marker='o', linestyle='-', alpha=0.7)
plt.plot(reconstructed_data_np[evento_idx, :num_features], label='Dato Ricostruito (Output)', color='red', marker='x', linestyle='--', alpha=0.7)

plt.title(f"Confronto Originale vs Ricostruito (Prime {num_features} feature dell'Evento)", fontsize=14)
plt.xlabel("Indice della Feature (Variabile cinematica)", fontsize=12)
plt.ylabel("Valore Normalizzato", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

import matplotlib.pyplot as plt
import numpy as np
import torch

# Mettiamo il modello in modalità valutazione (disabilita calcoli inutili)
model.eval()

# Prendiamo un intero blocco di validazione per fare i grafici
sample_data, _ = next(iter(val_loader))
sample_data = sample_data.to(device)

with torch.no_grad():
    # 1. Passiamo i dati SOLO nell'Encoder per ottenere lo spazio latente a 8D
    latent_space = model.get_latent(sample_data)

    # 2. Passiamo i dati in tutto l'Autoencoder per la ricostruzione
    reconstructed_data = model(sample_data)

# Portiamo i dati sulla CPU e li convertiamo in Numpy per i grafici
sample_data_np = sample_data.cpu().numpy()
latent_space_np = latent_space.cpu().numpy()
reconstructed_data_np = reconstructed_data.cpu().numpy()

# ==========================================
# PLOT 1: LO SPAZIO LATENTE (Le 8 Dimensioni)
# ==========================================
plt.figure(figsize=(16, 8))
plt.suptitle("Distribuzione delle 8 variabili nello Spazio Latente (Background)", fontsize=16, fontweight='bold')

for i in range(8):
    plt.subplot(2, 4, i+1)
    # Disegniamo l'istogramma per ogni singola dimensione latente
    plt.hist(latent_space_np[:, i], bins=50, color='teal', alpha=0.7, edgecolor='black')
    plt.title(f"Dimensione Latente {i+1}")
    plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

# ==========================================
# PLOT 2: ORIGINALE vs RICOSTRUITO
# ==========================================
# Prendiamo il primissimo evento del batch e mostriamo solo le prime 60 feature (su 2100)
evento_idx = 0
num_features = 60

plt.figure(figsize=(15, 5))
plt.plot(sample_data_np[evento_idx, :num_features], label='Dato Originale (Input)', color='blue', marker='o', linestyle='-', alpha=0.7)
plt.plot(reconstructed_data_np[evento_idx, :num_features], label='Dato Ricostruito (Output)', color='red', marker='x', linestyle='--', alpha=0.7)

plt.title(f"Confronto Originale vs Ricostruito (Prime {num_features} feature dell'Evento)", fontsize=14)
plt.xlabel("Indice della Feature (Variabile cinematica)", fontsize=12)
plt.ylabel("Valore Normalizzato", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

salvataggio spazio latene

In [ ]:
import torch
import numpy as np
import os

print("Inizio estrazione dello Spazio Latente (8D)...")

# Assicuriamoci che il modello sia in modalità valutazione
model.eval()

def extract_latent_space(dataloader):
    latent_vectors = []
    with torch.no_grad():
        for batch_x, _ in dataloader:
            batch_x = batch_x.to(device)
            # Usiamo SOLO l'encoder!
            latent = model.get_latent(batch_x)
            latent_vectors.append(latent.cpu().numpy())

    # Uniamo tutti i batch in un unico grande array numpy
    return np.vstack(latent_vectors)

# Per l'Anomaly Detection, abbiamo bisogno del background intero (train + val)
# Creiamo un DataLoader unico senza shuffle per estrarlo tutto in ordine
X_bg_full_tensor = torch.tensor(X_bg_scaled, dtype=torch.float32)
bg_full_loader = DataLoader(TensorDataset(X_bg_full_tensor, X_bg_full_tensor), batch_size=256, shuffle=False)

# Estraiamo!
latent_bg = extract_latent_space(bg_full_loader)
latent_test = extract_latent_space(test_loader)

print(f"Dimensioni Background Latente: {latent_bg.shape} (dovrebbe essere N, 8)")
print(f"Dimensioni BlackBox Latente: {latent_test.shape} (dovrebbe essere N, 8)")

# Salviamo questi nuovi dataset leggeri, saranno il punto di partenza per il Quantum Kernel!
save_dir = '/content/drive/MyDrive/set dati machine learning/processed_data'
np.save(os.path.join(save_dir, 'latent_bg.npy'), latent_bg)
np.save(os.path.join(save_dir, 'latent_blackbox.npy'), latent_test)

print("Dataset latenti salvati con successo su Drive!")

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import os

print("Calcolo dell'Errore di Ricostruzione (MSE) come Anomaly Score...")

# Mettiamo il modello in modalità valutazione
model.eval()
reconstruction_scores = []

# Calcolo MSE per ogni evento senza calcolare i gradienti
with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)

        # 1. La rete comprime e ricostruisce
        reconstructed = model(batch_x)

        # 2. Calcoliamo la differenza (Errore Quadratico Medio) tra l'input originale e l'output
        # La media viene fatta sulle colonne (dim=1), ottenendo un singolo score per evento
        mse_per_event = torch.mean((batch_x - reconstructed)**2, dim=1)

        # Salviamo i risultati convertendoli in numpy
        reconstruction_scores.extend(mse_per_event.cpu().numpy())

reconstruction_scores = np.array(reconstruction_scores)

# Carichiamo le label dal masterkey per le prime 20.000 righe
save_dir = '/content/drive/MyDrive/set dati machine learning/processed_data'
masterkey_path = os.path.join(save_dir, "events_LHCO2020_BlackBox1.masterkey")
true_labels_full = np.loadtxt(masterkey_path)
true_labels = true_labels_full[:len(reconstruction_scores)]

# Calcolo ROC e AUC
fpr, tpr, thresholds = roc_curve(true_labels, reconstruction_scores)
roc_auc = auc(fpr, tpr)

print(f"\n---> PRESTAZIONE CON ERRORE DI RICOSTRUZIONE (AUC): {roc_auc:.4f}")

# Visualizzazione dell'istogramma degli errori
plt.figure(figsize=(10, 6))
# Istogramma del background (label 0)
plt.hist(reconstruction_scores[true_labels == 0], bins=100, color='royalblue', alpha=0.6, density=True, label='Eventi Normali')
# Istogramma delle anomalie (label 1)
plt.hist(reconstruction_scores[true_labels == 1], bins=100, color='crimson', alpha=0.8, density=True, label='Vere Anomalie')

plt.title(f"Anomaly Score (Errore di Ricostruzione MSE) - AUC: {roc_auc:.3f}", fontsize=14, fontweight='bold')
plt.xlabel("Errore di Ricostruzione Quadratico Medio", fontsize=12)
plt.ylabel("Densità", fontsize=12)
# Usiamo scala logaritmica sull'asse X per via di possibili outlier molto grandi
plt.xscale('log')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

from sklearn.preprocessing import StandardScaler

# Normalizziamo lo spazio latente prima del clustering
scaler_l = StandardScaler()
latent_bg_norm = scaler_l.fit_transform(latent_bg)
latent_blackbox_norm = scaler_l.transform(latent_blackbox)

# K-Means sullo spazio normalizzato
k_clusters = 8 # <-- Proviamo ad aumentare i cluster da 4 a 8 (o anche 10)
print(f"\nAddestramento di K-Means con {k_clusters} cluster...")
kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init='auto')
kmeans.fit(latent_bg_norm)

distances_to_clusters = kmeans.transform(latent_blackbox_norm)
anomaly_scores = np.min(distances_to_clusters, axis=1)

# Calcolo ROC e AUC con logica di inversione
fpr, tpr, thresholds = roc_curve(true_labels, anomaly_scores)
roc_auc = auc(fpr, tpr)

# Se l'AUC è sotto 0.5, il modello funziona ma al contrario! Lo sistemiamo:
if roc_auc < 0.5:
    print(f"---> Ribalto l'AUC...")
    roc_auc = 1 - roc_auc
    anomaly_scores = -anomaly_scores # Invertiamo per il grafico

print(f"\n---> PRESTAZIONE MODELLO (K-Means AUC REALE): {roc_auc:.4f}")

fig = plt.figure(figsize=(18, 7))

# --- PLOT 1: Spazio 3D con Centroidi K-Means ---
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(true_normal_3d[:sample_size_norm, 0], true_normal_3d[:sample_size_norm, 1], true_normal_3d[:sample_size_norm, 2],
            c='royalblue', alpha=0.1, label='Normali (Label 0)', s=10)
ax1.scatter(true_anomalies_3d[:, 0], true_anomalies_3d[:, 1], true_anomalies_3d[:, 2],
            c='crimson', alpha=0.8, label='Anomalie (Label 1)', s=25, edgecolors='black')

# Disegniamo i centri dei cluster K-Means
ax1.scatter(cluster_centers_3d[:, 0], cluster_centers_3d[:, 1], cluster_centers_3d[:, 2],
            c='yellow', marker='*', s=350, edgecolors='black', label='Centri Cluster K-Means')

ax1.set_title("Spazio Latente con K-Means (PCA 3D)", fontsize=14, fontweight='bold')
ax1.legend(loc='upper right')

# --- PLOT 2: Distribuzione dell'Anomaly Score ---
ax2 = fig.add_subplot(122)
ax2.hist(anomaly_scores[true_labels == 0], bins=80, color='royalblue', alpha=0.6, density=True, label='Eventi Normali')
ax2.hist(anomaly_scores[true_labels == 1], bins=80, color='crimson', alpha=0.8, density=True, label='Vere Anomalie')

ax2.set_title(f"Distribuzione Anomaly Score K-Means (AUC = {roc_auc:.3f})", fontsize=14, fontweight='bold')
ax2.set_xlabel("Distanza dal Cluster più vicino (Nello spazio 8D)", fontsize=12)
ax2.set_ylabel("Densità", fontsize=12)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

print("1. Ricarico i dati RAW della BlackBox per i calcoli fisici...")
# Riprendiamo i dati prima di qualsiasi preprocessing!
X_test_raw = np.load('/content/drive/MyDrive/set dati machine learning/processed_data/blackbox_subset.npy')

def calcola_massa_invariante_evento(dati_raw):
    

    Converte pT, eta, phi in 4-momento (E, px, py, pz) assumendo particelle a massa zero.
    Somma i 4-momenti di tutte le particelle e calcola la massa invariante dell'evento.
    

In [ ]:
    N_eventi, N_feature = dati_raw.shape
    N_particelle = N_feature // 3
    dati_3d = dati_raw.reshape(N_eventi, N_particelle, 3)

    pt = dati_3d[:, :, 0]
    eta = dati_3d[:, :, 1]
    phi = dati_3d[:, :, 2]

    # Conversione in coordinate cartesiane e calcolo Energia
    px = pt * np.cos(phi)
    py = pt * np.sin(phi)
    pz = pt * np.sinh(eta)
    E = pt * np.cosh(eta)

    # Somma totale sull'evento
    Px_tot = np.sum(px, axis=1)
    Py_tot = np.sum(py, axis=1)
    Pz_tot = np.sum(pz, axis=1)
    E_tot = np.sum(E, axis=1)

    # M = sqrt(E^2 - P^2)
    M2 = E_tot**2 - Px_tot**2 - Py_tot**2 - Pz_tot**2
    return np.sqrt(np.maximum(M2, 0)) # maximum evita errori di arrotondamento < 0

print("2. Calcolo della Massa Invariante per tutti gli eventi...")
masse_blackbox = calcola_massa_invariante_evento(X_test_raw)

# ---------------------------------------------------------
# IL TAGLIO FISICO (MASS WINDOW)
# ---------------------------------------------------------
# Nel dataset LHCO le energie sono spesso in GeV. 3.8 TeV = 3800 GeV.
# Creiamo una finestra stretta per escludere il rumore: es. da 3200 a 4400.
finestra_min = 3200
finestra_max = 4400

# Creiamo una maschera booleana (True se dentro la finestra, False se fuori)
maschera_massa = (masse_blackbox >= finestra_min) & (masse_blackbox <= finestra_max)
eventi_selezionati = np.sum(maschera_massa)

print(f"\n3. Applicazione Taglio di Massa: {finestra_min} < M < {finestra_max} GeV")
print(f"Eventi trattenuti: {eventi_selezionati} su {len(masse_blackbox)} ({eventi_selezionati/len(masse_blackbox)*100:.1f}%)")

# ---------------------------------------------------------
# CALCOLO NUOVA AUC (Sui dati filtrati)
# ---------------------------------------------------------
# Usiamo la maschera per tagliare le label e gli score calcolati in precedenza.
# (Assicurati di aver in memoria 'anomaly_scores' dal K-Means o GMM,
#  oppure 'reconstruction_scores' dall'Autoencoder)

labels_filtrate = true_labels[maschera_massa]

# TEST SULLO SCORE K-MEANS / GMM (Sostituisci la variabile con il tuo score migliore attuale)
scores_filtrati = anomaly_scores[maschera_massa]

fpr_f, tpr_f, _ = roc_curve(labels_filtrate, scores_filtrati)
auc_f = auc(fpr_f, tpr_f)

# Ribaltamento nel caso l'anomalia sia "meno anomala" del fondo
if auc_f < 0.5:
    auc_f = 1 - auc_f

print(f"\n===> PRESTAZIONE CON TAGLIO DI MASSA (AUC): {auc_f:.4f} <===")

# ---------------------------------------------------------
# PLOT DELLA MASSA INVARIANTE
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.hist(masse_blackbox[true_labels == 0], bins=100, range=(0, 8000), alpha=0.6, color='royalblue', label='Background (Normale)', density=True)
plt.hist(masse_blackbox[true_labels == 1], bins=100, range=(0, 8000), alpha=0.8, color='crimson', label='Anomalie (Z\')', density=True)

# Disegniamo la finestra
plt.axvline(finestra_min, color='black', linestyle='--', linewidth=2, label='Finestra di Massa')
plt.axvline(finestra_max, color='black', linestyle='--', linewidth=2)

plt.title("Spettro della Massa Invariante", fontsize=14, fontweight='bold')
plt.xlabel("Massa Invariante (GeV)", fontsize=12)
plt.ylabel("Densità", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()